## 1、流式调用 、非流式调用

In [4]:
# 非流式调用
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, SystemMessage, HumanMessage
import os
import dotenv
from openai import max_retries

dotenv.load_dotenv()
# 1、获取大模型的实例
model = init_chat_model(
    model=os.getenv("MODEL_NAME"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    temperature = 0.5,
)
res = model.invoke("你好，你是谁")
print(res)

content='你好！我是一个由OpenAI开发的人工智能语言模型，叫做ChatGPT。很高兴为你提供帮助！你有什么问题或者需要聊些什么吗？' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 10, 'total_tokens': 47, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DZCQFx3Jy1fAGCQSYpPKUH1qGIwk1', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019dce3d-0f79-7a91-9c07-6bef48dd0ac5-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 10, 'output_tokens': 37, 'total_tokens': 47, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [5]:
# 流式调用，通过model.stream方法去调用，
# 返回的是一个生成器，通过迭代生成器的方式，得到结果
res = model.stream("你好，你是谁")

In [6]:
for chunk in res:
    print(chunk.content,end="")

你好！我是一个由OpenAI开发的人工智能语言模型，叫做ChatGPT。很高兴为你提供帮助！你有什么问题或者需要聊些什么吗？

## 2、批次调用、非批次调用

In [7]:
# 批次调用，通过model.batch方法实现，底层原理就是通过多线程的方式去调用，
#
messages = [
    [
        {"role": "system", "content": "你是一位诗人"},
        {"role": "user", "content": "写一首关于春天的诗"},
    ],
    [
        {"role": "system", "content": "你是一位诗人"},
        {"role": "user", "content": "写一首关于夏天的诗"},
    ],
    [
        {"role": "system", "content": "你是一位诗人"},
        {"role": "user", "content": "写一首关于秋天的诗"},
    ],
]
res = model.batch(messages)

## 3、同步调用、异步调用

In [8]:
# 同步调用：多次请求之间串行处理，B请求需要A请求完成之后，再发出请求，得到响应
messagess = [
    [
        {"role": "system", "content": "你是一位诗人"},
        {"role": "user", "content": "写一首关于春天的诗"},
    ],
    [
        {"role": "system", "content": "你是一位诗人"},
        {"role": "user", "content": "写一首关于夏天的诗"},
    ],
    [
        {"role": "system", "content": "你是一位诗人"},
        {"role": "user", "content": "写一首关于秋天的诗"},
    ],
]
import time
start_time = time.time()
res = [model.invoke(messages) for messages in messagess]
end_time = time.time()
print(f"总耗时:{end_time - start_time}")

总耗时:7.178855895996094


In [9]:
# 异步调用：model.ainvoke方法，返回一个协程对象，把多个协程对象可以打包成一个协程对象，
# await最终的协程对象，就能够实现异步调用
# 异步调用，能够提高程序的性能。相对于batch调用而言，能够减少资源（线程数）使用量
import asyncio
async def gather_task(messages:list):
    # 调用ainvoke并不会真正地发起请求
    tasks = [model.ainvoke(message_list) for message_list in messages]
    return await asyncio.gather(*tasks)
gather_task(messagess)

<coroutine object gather_task at 0x000001F9FE9926B0>

In [12]:
await gather_task(messagess)
# asyncio.run(gather_task(messagess))

[AIMessage(content='春风拂面柳丝长，\n花开满园蝶舞翔。\n燕语呢喃新绿里，\n溪水潺潺唱希望。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 24, 'total_tokens': 60, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DZCe25RpbrlK89iBQnSBnAFm4JRCH', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dce4a-2079-7270-a3c5-f83a7991afdb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, 'output_tokens': 36, 'total_tokens': 60, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 AIMessage(content='夏日炎炎绿意浓，\n蝉鸣阵阵绕林中。\n荷叶田田映碧水，\n微风拂面送清风。\n\n夕阳西下染云霞，\n蛙声此起又彼和。\n繁星点点夜空亮，